# N24News — EDA for the A3 Dataset Proposal

Produces the statistics, figures, and split used in `dataproposal/dataproposal-a3.tex`.
Run on Google Colab (CPU runtime is enough). Outputs are saved to Google Drive (`MyDrive/DL_A3/eda_outputs`).

## 1. Setup

In [ ]:
!pip -q install gdown imagehash

In [ ]:
import sys, re, json, html, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import imagehash
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/n24news")
    OUT_DIR = Path("/content/drive/MyDrive/DL_A3/eda_outputs")
else:
    DATA_DIR = Path("n24news")
    OUT_DIR = Path("eda_outputs")

FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

GDRIVE_ID = "1OS1fXwZ1Vsj70lEQajccyssxQRYp5X9D"  # link in the official N24News README
SEED = 42
PHASH_MAX_DIST = 4  # Hamming distance (out of 64 bits) for near-identical images

sns.set_theme(style="whitegrid")

def save_fig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=200)
    plt.savefig(FIG_DIR / f"{name}.pdf")
    plt.show()

## 2. Download and load the data

If `gdown` fails with a quota error, open `https://drive.google.com/file/d/1OS1fXwZ1Vsj70lEQajccyssxQRYp5X9D`,
add it to your Drive, and copy `N24News.zip` into `DATA_DIR`.

In [ ]:
zip_path = DATA_DIR / "N24News.zip"
if not (DATA_DIR / "news" / "nytimes.json").exists():
    if not zip_path.exists():
        import gdown
        gdown.download(id=GDRIVE_ID, output=str(zip_path), quiet=False)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR)

print(sorted(p.name for p in (DATA_DIR / "news").iterdir()))

In [ ]:
raw = pd.read_json(DATA_DIR / "news" / "nytimes.json")
print(raw.shape)
print(raw.dtypes)

def clean(s):
    return s.fillna("").astype(str).map(html.unescape).str.replace(r"\s+", " ", regex=True).str.strip()

df = pd.DataFrame({
    "image_id": raw["image_id"].astype(str),
    "label": raw["section"],
    "headline": clean(raw["headline"]),
    "abstract": clean(raw["abstract"]),
    "caption": clean(raw["caption"]),
    "body": clean(raw["article"]),
    "url": raw["article_url"].astype(str),
})

# official split released by the authors
official = {}
for name in ["train", "dev", "test"]:
    for r in json.loads((DATA_DIR / "news" / f"nytimes_{name}.json").read_text(encoding="utf-8")):
        official[str(r["image_id"])] = name
df["official_split"] = df["image_id"].map(official)

print(df["official_split"].value_counts(dropna=False))
df.head(3)

## 3. Images: validity, resolution, perceptual hash

In [ ]:
IMG_DIR = next(DATA_DIR.rglob("*.jpg")).parent
print("image folder:", IMG_DIR, "| files:", len(list(IMG_DIR.glob("*.jpg"))))
df["img_path"] = [IMG_DIR / f"{i}.jpg" for i in df["image_id"]]

def check_image(path):
    # returns width, height, mode, phash (None if the file is missing or unreadable)
    try:
        with Image.open(path) as im:
            im.load()
            return im.size[0], im.size[1], im.mode, str(imagehash.phash(im.convert("RGB")))
    except Exception:
        return None, None, None, None

# this pass takes a while, so the result is cached on Drive
cache = OUT_DIR / "image_check.csv"
if cache.exists():
    img = pd.read_csv(cache, dtype={"image_id": str, "phash": str})
else:
    rows = [check_image(p) for p in tqdm(df["img_path"])]
    img = pd.DataFrame(rows, columns=["width", "height", "mode", "phash"])
    img.insert(0, "image_id", df["image_id"].values)
    img.to_csv(cache, index=False)

df = df.merge(img[["image_id", "width", "height", "mode", "phash"]], on="image_id", how="left")
df["img_ok"] = df["phash"].notna()
print("unreadable or missing images:", (~df["img_ok"]).sum())

In [ ]:
ok = df[df["img_ok"]]
aspect = ok["width"] / ok["height"]
image_stats = pd.DataFrame({
    "width": ok["width"].describe(percentiles=[.05, .5, .95]),
    "height": ok["height"].describe(percentiles=[.05, .5, .95]),
    "aspect (w/h)": aspect.describe(percentiles=[.05, .5, .95]),
}).round(2)
image_stats.to_csv(OUT_DIR / "image_stats.csv")
print(image_stats)
print("colour modes:", ok["mode"].value_counts().to_dict())
print("shorter side < 224 px:", (ok[["width", "height"]].min(axis=1) < 224).sum())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(ok["width"], bins=60, alpha=.7, label="width")
axes[0].hist(ok["height"], bins=60, alpha=.7, label="height")
axes[0].set_xlabel("pixels"); axes[0].set_title("Image resolution"); axes[0].legend()
axes[1].hist(aspect.clip(0, 3), bins=60)
axes[1].set_xlabel("width / height"); axes[1].set_title("Aspect ratio")
save_fig("image_resolution")

## 4. Pair validity and missing modalities

In [ ]:
validity = pd.Series({
    "pairs": len(df),
    "missing or unreadable image": (~df["img_ok"]).sum(),
    "empty headline": (df["headline"] == "").sum(),
    "empty abstract": (df["abstract"] == "").sum(),
    "empty caption": (df["caption"] == "").sum(),
    "empty body": (df["body"] == "").sum(),
    "missing label": df["label"].isna().sum(),
})
df["valid"] = df["img_ok"] & ((df["headline"] != "") | (df["abstract"] != "")) & df["label"].notna()
validity["valid pairs"] = df["valid"].sum()
validity.to_frame("count").to_csv(OUT_DIR / "validity.csv")
print(validity)

dv = df[df["valid"]].reset_index(drop=True)

## 5. Class distribution

In [ ]:
counts = dv["label"].value_counts()
class_dist = pd.DataFrame({"count": counts, "percent": (100 * counts / counts.sum()).round(2)})
class_dist.to_csv(OUT_DIR / "class_distribution_all.csv")
print(class_dist)
print(f"{len(counts)} classes, largest/smallest = {counts.max() / counts.min():.2f}")

## 6. Text length

In [ ]:
dv["text"] = dv["headline"] + " " + dv["abstract"]
fields = ["headline", "abstract", "text", "caption", "body"]
for f in fields:
    dv[f"words_{f}"] = dv[f].str.split().str.len()

text_stats = pd.DataFrame({
    f: dv[f"words_{f}"].describe(percentiles=[.5, .95])[["mean", "50%", "95%", "max"]] for f in fields
}).T.round(2)
text_stats.to_csv(OUT_DIR / "text_stats.csv")
print(text_stats)

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("roberta-base")

dv["tok_headline"] = [len(x) for x in tok(dv["headline"].tolist(), add_special_tokens=False)["input_ids"]]
dv["tok_abstract"] = [len(x) for x in tok(dv["abstract"].tolist(), add_special_tokens=False)["input_ids"]]
dv["tok_pair"] = [len(x) for x in tok(dv["headline"].tolist(), dv["abstract"].tolist())["input_ids"]]  # with special tokens
print(dv[["tok_headline", "tok_abstract", "tok_pair"]].describe(percentiles=[.5, .95, .99]).round(2))
print("pairs longer than 128 tokens:", (dv["tok_pair"] > 128).sum())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, f, title in zip(axes, ["words_headline", "words_abstract", "words_text"],
                        ["Headline", "Abstract", "Headline + abstract"]):
    ax.hist(dv[f], bins=range(0, int(dv[f].quantile(.995)) + 2))
    ax.axvline(dv[f].median(), color="k", ls="--", lw=1, label=f"median={dv[f].median():.0f}")
    ax.set_title(title); ax.set_xlabel("words"); ax.legend()
axes[0].set_ylabel("articles")
save_fig("text_length")

## 7. Duplicates and leakage groups

* near-identical images: perceptual hashes within `PHASH_MAX_DIST` bits
* repeated headlines: same headline after lower-casing and removing punctuation

Both relations are edges of a graph; each connected component is one duplicate group, which must stay in one split.

In [ ]:
hashes = np.array([int(h, 16) for h in dv["phash"]], dtype=np.uint64)
image_pairs = []
for start in tqdm(range(0, len(hashes), 500)):
    dist = np.bitwise_count(hashes[start:start + 500, None] ^ hashes[None, :])
    a, b = np.nonzero(dist <= PHASH_MAX_DIST)
    a = a + start
    image_pairs += [(i, j) for i, j in zip(a, b) if i < j]

print("near-identical image pairs:", len(image_pairs))
print("articles sharing an identical phash:", dv["phash"].duplicated(keep=False).sum())

In [ ]:
dv["headline_norm"] = (dv["headline"].str.lower()
                       .str.replace(r"[^a-z0-9 ]+", " ", regex=True)
                       .str.replace(r"\s+", " ", regex=True).str.strip())
headline_pairs = []
for _, idx in dv.groupby("headline_norm").groups.items():
    idx = list(idx)
    headline_pairs += [(idx[0], j) for j in idx[1:]]

print("articles with a repeated headline:", dv["headline_norm"].duplicated(keep=False).sum())
print(dv["headline_norm"].value_counts().head(10))

In [ ]:
G = nx.Graph()
G.add_nodes_from(range(len(dv)))
G.add_edges_from(image_pairs + headline_pairs)
group_of = {}
for gid, comp in enumerate(nx.connected_components(G)):
    for i in comp:
        group_of[i] = gid
dv["group_id"] = dv.index.map(group_of)

group_size = dv["group_id"].map(dv["group_id"].value_counts())
dv["in_dup_group"] = group_size > 1
labels_per_group = dv[dv["in_dup_group"]].groupby("group_id")["label"].nunique()
conflict_groups = labels_per_group[labels_per_group > 1].index

duplicates = pd.Series({
    "articles sharing an identical phash": dv["phash"].duplicated(keep=False).sum(),
    "near-identical image pairs": len(image_pairs),
    "articles in near-identical image pairs": len({i for p in image_pairs for i in p}),
    "articles with a repeated headline": dv["headline_norm"].duplicated(keep=False).sum(),
    "duplicate groups (size > 1)": (dv["group_id"].value_counts() > 1).sum(),
    "articles in duplicate groups": dv["in_dup_group"].sum(),
    "largest duplicate group": group_size.max(),
    "groups with conflicting labels": len(conflict_groups),
    "articles in conflicting groups": dv["group_id"].isin(conflict_groups).sum(),
})
duplicates.to_frame("count").to_csv(OUT_DIR / "duplicates.csv")
print(duplicates)

In [ ]:
# a few of the largest duplicate groups, to see what they look like
fig, axes = plt.subplots(4, 4, figsize=(13, 8))
for row, gid in zip(axes, dv.loc[dv["in_dup_group"], "group_id"].value_counts().index[:4]):
    members = dv[dv["group_id"] == gid].head(4)
    for ax in row:
        ax.axis("off")
    for ax, (_, r) in zip(row, members.iterrows()):
        ax.imshow(Image.open(r["img_path"]).convert("RGB"))
        ax.set_title(f"{r['label']}\n{r['headline'][:40]}", fontsize=8)
save_fig("duplicate_examples")

## 8. Publication period (date in the article URL)

In [ ]:
date_parts = dv["url"].str.extract(r"/(\d{4})/(\d{2})/(\d{2})/")
dv["date"] = pd.to_datetime(date_parts[0] + "-" + date_parts[1] + "-" + date_parts[2], errors="coerce")
print("dated articles:", dv["date"].notna().sum(), "/", len(dv))
print("period:", dv["date"].min().date(), "->", dv["date"].max().date())

dv["date"].dt.to_period("M").value_counts().sort_index().plot(figsize=(11, 3), title="Articles per month")
save_fig("publication_period")

## 9. Duplicate-aware stratified 80/10/10 split

`StratifiedGroupKFold` with 10 folds: fold 0 = test, fold 1 = val, the rest = train.
We also measure how many test articles would have a duplicate in train under a random split and under the official split.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
fold = np.zeros(len(dv), dtype=int)
for k, (_, idx) in enumerate(sgkf.split(dv, dv["label"], dv["group_id"])):
    fold[idx] = k
dv["split"] = np.select([fold == 0, fold == 1], ["test", "val"], "train")
assert dv.groupby("group_id")["split"].nunique().max() == 1  # no group crosses splits

def leakage(split_col, eval_name):
    train_groups = set(dv.loc[dv[split_col] == "train", "group_id"])
    ev = dv[dv[split_col] == eval_name]
    leaked = (ev["in_dup_group"] & ev["group_id"].isin(train_groups)).sum()
    print(f"{split_col:15s} {eval_name:5s}: {leaked} / {len(ev)} ({100 * leaked / len(ev):.2f}%) have a duplicate in train")
    return leaked, len(ev)

_, test_idx = train_test_split(dv.index, test_size=0.1, stratify=dv["label"], random_state=SEED)
dv["random_split"] = "train"
dv.loc[test_idx, "random_split"] = "test"

leak = {
    "random test": leakage("random_split", "test"),
    "official dev": leakage("official_split", "dev"),
    "official test": leakage("official_split", "test"),
    "ours val": leakage("split", "val"),
    "ours test": leakage("split", "test"),
}

In [ ]:
per_split = pd.crosstab(dv["label"], dv["split"])[["train", "val", "test"]]
per_split.to_csv(OUT_DIR / "class_distribution_per_split.csv")
print(dv["split"].value_counts())
print(dv.groupby("split")["in_dup_group"].sum())
print(per_split)

dv[["image_id", "label", "split", "group_id"]].to_csv(OUT_DIR / "splits.csv", index=False)

In [ ]:
order = per_split.sort_values("train").index
fig, axes = plt.subplots(1, 2, figsize=(12, 7), sharey=True)
axes[0].barh(order, per_split.loc[order, "train"], color="#4C72B0"); axes[0].set_title("Train: articles per category")
axes[1].barh(order, per_split.loc[order, "test"], color="#DD8452"); axes[1].set_title("Test: articles per category")
for ax in axes:
    ax.set_xlabel("articles")
save_fig("class_distribution")

## 10. Example image–text pairs

In [ ]:
examples = dv.groupby("label").head(1).sample(8, random_state=SEED)
fig, axes = plt.subplots(2, 4, figsize=(15, 4.6))
for ax, (_, r) in zip(axes.ravel(), examples.iterrows()):
    ax.imshow(Image.open(r["img_path"]).convert("RGB")); ax.axis("off")
    ax.set_title(f"[{r['label']}]\n{r['headline'][:45]}", fontsize=9)
save_fig("sample_pairs")

## 11. Summary for the proposal

In [ ]:
summary = {
    "pairs": int(len(df)),
    "valid_pairs": int(len(dv)),
    "validity": {k: int(v) for k, v in validity.items()},
    "n_classes": int(dv["label"].nunique()),
    "class_counts": {k: int(v) for k, v in counts.items()},
    "imbalance_ratio": round(float(counts.max() / counts.min()), 2),
    "text_words": text_stats.to_dict(orient="index"),
    "pair_tokens_p95_p99_max": [float(dv["tok_pair"].quantile(.95)), float(dv["tok_pair"].quantile(.99)), int(dv["tok_pair"].max())],
    "image_median_wh": [float(ok["width"].median()), float(ok["height"].median())],
    "duplicates": {k: int(v) for k, v in duplicates.items()},
    "leakage": {k: {"leaked": int(a), "total": int(b)} for k, (a, b) in leak.items()},
    "split_sizes": {k: int(v) for k, v in dv["split"].value_counts().items()},
    "period": [str(dv["date"].min().date()), str(dv["date"].max().date())],
}
(OUT_DIR / "eda_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))